# 01 · Prepare the data  (run ONCE, by Dawar)
**Kaggle settings (right panel):** Accelerator = **GPU T4 x1** (or P100), Internet = **On**.  Run time: about 30-40 min.

What this notebook does (Milestone 2, Steps 1-4):
1. Loads the ARC-Easy test set (4-option items) and 1,000 training items for the format-only control.
2. Checks every test item against the Pile with the infini-gram API and drops items Pythia has already seen.
3. Scores the base Pythia-410M model on every item (difficulty and base detector scores).
4. Assigns items to the 9 contaminated cells (150 each), 450 clean items and 300 FSD reference items, matched on difficulty and length.
5. Builds near-duplicates and generates paraphrases with Qwen2.5-3B-Instruct (filtered by similarity >= 0.85 and no shared 8-word sequence).

**Output:** `outputs_01.zip` (in the Output panel on the right). Download it, unzip it **into your local repo folder** (it contains `data/` and `results/`), then double-click `commit_and_push.bat`. Notebook 02 reads these files from GitHub.

In [4]:
import io
import sys
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from zipfile import ZipFile

REPO_URL = "https://github.com/DawarAhmed27/DAV-Reseach-DataContamination"
REF = "main"
ARCHIVE_URL = f"{REPO_URL}/archive/refs/heads/{REF}.zip"

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
try:
    with urlopen(ARCHIVE_URL, timeout=60) as response:
        archive = response.read()
except URLError as exc:
    raise RuntimeError(f"Could not download {ARCHIVE_URL}: {exc}") from exc

# Extract each download to its own directory to keep earlier runs available.
from tempfile import mkdtemp
download = Path(mkdtemp(prefix="dav_github_", dir=WORK))
with ZipFile(io.BytesIO(archive)) as project:
    root_name = project.namelist()[0].split("/")[0]
    project.extractall(download)
REPO = download / root_name

required = ['dcl/pipeline.py']
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(f"Files missing from GitHub branch {REF}: {', '.join(missing)}")

sys.path.insert(0, str(REPO))
print("Downloaded project from GitHub:", REPO)


Downloaded project from GitHub: C:\Users\armug\Documents\FAST NUCES\FAST NUCES\FAST Sem 5\DAV\Research\Implimentation\Implimentation\notebooks\dav_github_jw740dpg\DAV-Reseach-DataContamination-main


In [ ]:
# --- Libraries that Kaggle does not always have ---
!pip uninstall -y -q torchao 2>/dev/null   # Kaggle's old torchao breaks the latest peft; we do not use it
!pip install -q peft sentence-transformers 2>/dev/null | tail -1
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4 in Settings > Accelerator")

The system cannot find the path specified.
The system cannot find the path specified.


In [ ]:
from dcl import pipeline as P
items, base, log = P.prepare_data(out_root=REPO)
log

In [ ]:
# Quick checks: group sizes and difficulty balance (should be similar across groups)
import pandas as pd
print(items["group"].value_counts().sort_index().to_string())
print(items.groupby(items["group"].str.replace(r"_x\d+", "", regex=True))["base_p_correct"].mean().round(3).to_string())

In [ ]:
# Package the outputs for download
import shutil, zipfile
zpath = WORK / "outputs_01.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in list((REPO / "data").glob("*")) + [REPO / "results" / "base.csv"]:
        if p.is_file() and p.name != ".gitkeep":
            z.write(p, p.relative_to(REPO))
print("Download", zpath, "from the Output panel, unzip into your repo folder, then run commit_and_push.bat")

**Before notebook 02:** open `data/paraphrase_manual_check.csv`, fill the column `meaning_kept (Y/N)` for the 50 rows (Armughan can do this), and commit it. The result goes in the report.